# Original Two-Hour Mock Assessment

Audience: Python learners preparing for quant interviews. Prerequisites: basic Python and pandas.

Goals: 120 minutes: validation, joins, quotes, accounting, risk.

Outline: setup, short tasks, checks, pitfalls. Run cells from top to bottom. All market data here is synthetic.


**Timebox: 120 minutes.** This is an original practice assessment. No live or proprietary material is used. Public sample checks are outside this notebook; run `python scripts/check_exercises.py 05` after filling TODOs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from market_simulator import (MarketMaker, ClientModel, MarketEnvironment,
    QuotedTrade, CompletedTrade, generate_market_data, generate_rfqs, run_simulation, make_quote)
from market_simulator.risk import beta, historical_volatility

# Resolve synthetic data from the installed local package, independent of kernel CWD.
import market_simulator
DATA = Path(market_simulator.__file__).resolve().parents[1] / 'data'
prices = pd.read_csv(DATA / 'market_prices.csv', parse_dates=['timestamp'])
requests = pd.read_csv(DATA / 'client_rfqs.csv', parse_dates=['timestamp'])
print(prices.head(2).to_string(index=False))


## Part 1 — Validate (15 min)

Return a clean copy of a market DataFrame with columns timestamp,ticker,mid. Remove rows with missing keys or nonpositive/nonfinite mid; keep the latest row for duplicate (timestamp,ticker) keys. Never mutate input. Example duplicate key -> one retained row.


In [ ]:
def clean_market(frame: pd.DataFrame) -> pd.DataFrame:
    required = {'timestamp','ticker','mid'}
    if not required.issubset(frame.columns):
        raise ValueError('missing columns')
    result = frame.copy()
    result = result.dropna(subset=['timestamp','ticker','mid'])
    result = result.loc[np.isfinite(result['mid']) & (result['mid'] > 0)]
    return result.drop_duplicates(['timestamp','ticker'], keep='last').reset_index(drop=True)

**Reasoning and complexity:** O(n) expected time and space. Copy before filtering; keep-last is intentional.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
d = pd.DataFrame({'timestamp':[1,1,2], 'ticker':['A','A','B'], 'mid':[100,101,-2]})
assert clean_market(d).mid.tolist() == [101]

## Part 2 — Match requests (20 min)

For each RFQ (timestamp,ticker,quantity), attach the matching mid from cleaned market. Preserve request order; raise ValueError on missing prices or nonpositive quantities. Return DataFrame with original columns plus mid.


In [ ]:
def match_requests(rfqs: pd.DataFrame, market: pd.DataFrame) -> pd.DataFrame:
    if (rfqs['quantity'] <= 0).any() or rfqs['quantity'].isna().any():
        raise ValueError('invalid quantity')
    if market.duplicated(['timestamp','ticker']).any():
        raise ValueError('ambiguous market key')
    result = rfqs.merge(market[['timestamp','ticker','mid']], on=['timestamp','ticker'], how='left', sort=False, validate='many_to_one')
    if result['mid'].isna().any():
        raise ValueError('missing price')
    return result

**Reasoning and complexity:** O(n+m) expected for a hash join. Many-to-one validation catches duplicate market keys.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert len(match_requests(requests.head(2), clean_market(prices))) == 2

## Part 3 — Quote (25 min)

For a positive mid and signed inventory, return (bid,ask). Full spread is 2% of mid. Shift center down by 0.5% of mid for every 50 long shares, up for short shares; cap inventory magnitude at 100 for skew. Example mid 100, inventory 0 -> (99,101).


In [ ]:
def quote_with_inventory(mid: float, inventory: int) -> tuple[float, float]:
    if not np.isfinite(mid) or mid <= 0:
        raise ValueError('invalid mid')
    clipped = max(-100, min(100, inventory))
    center = mid * (1 - .005 * clipped / 50)
    return center - .01*mid, center + .01*mid

**Reasoning and complexity:** O(1). Skew shifts the quote center, not its width; cap prevents extreme inventory from making prices invalid.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert quote_with_inventory(100,0) == (99,101)
assert quote_with_inventory(100,50)[1] < 101

## Part 4 — Apply fills (30 min)

Process rows with side ('buy' or 'sell' from maker viewpoint), quantity, price. Return (inventory,cash), starting at zero inventory and given initial_cash. Buy spends cash; sell earns cash. Reject invalid rows; do not mutate input.


In [ ]:
def replay_fills(fills: pd.DataFrame, initial_cash: float) -> tuple[int, float]:
    position, cash = 0, float(initial_cash)
    for row in fills.itertuples(index=False):
        if row.side not in {'buy','sell'} or row.quantity <= 0 or row.price <= 0:
            raise ValueError('invalid fill')
        delta = row.quantity if row.side == 'buy' else -row.quantity
        position += delta
        cash -= delta * row.price
    return position, cash

**Reasoning and complexity:** O(n) time and O(1) extra space. Cash is not profit while open inventory remains.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
f = pd.DataFrame([{'side':'buy','quantity':2,'price':99},{'side':'sell','quantity':1,'price':101}])
assert replay_fills(f, 1000) == (1,903.0)

## Part 5 — Risk and P&L (15 min)

Given cash, signed positions and current positive marks, return a dict with `equity`, `net_exposure`, and `gross_exposure`. Equity is cash plus signed marked inventory; gross exposure sums absolute marked inventory. Example cash 1000, A:2 at 100 -> equity 1200, net/gross 200.


In [ ]:
def risk_report(cash: float, positions: dict[str, int], marks: dict[str, float]) -> dict[str, float]:
    values = []
    for ticker, quantity in positions.items():
        price = marks[ticker]
        if not np.isfinite(price) or price <= 0:
            raise ValueError('invalid mark')
        values.append(quantity * price)
    net = sum(values)
    return {'equity': cash + net, 'net_exposure': net, 'gross_exposure': sum(abs(v) for v in values)}

**Reasoning and complexity:** O(k) time and O(k) space. Net exposure can cancel while gross remains large.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert risk_report(1000, {'A':2}, {'A':100}) == {'equity':1200,'net_exposure':200,'gross_exposure':200}

## Part 6 — Beta hedge (15 min)

Given aligned asset and benchmark return Series, a signed asset position, and positive current prices, estimate beta = covariance(asset, benchmark)/variance(benchmark). Return signed integer benchmark units to offset the asset's beta-weighted dollar exposure, rounded with `round`. Reject fewer than two aligned returns, zero benchmark variance, or invalid prices. Example beta 2, asset 10 shares at 50, benchmark at 100 -> -10 units.


In [ ]:
def beta_hedge_units(asset_returns: pd.Series, benchmark_returns: pd.Series, asset_position: int, asset_price: float, benchmark_price: float) -> int:
    if asset_price <= 0 or benchmark_price <= 0:
        raise ValueError('invalid price')
    hedge_beta = beta(asset_returns, benchmark_returns)
    return round(-asset_position * asset_price * hedge_beta / benchmark_price)

**Reasoning and complexity:** O(n) for aligned returns; O(n) temporary space. This beta hedge targets broad co-movement, not idiosyncratic risk or execution costs.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
r = pd.Series([.01,-.02,.03])
assert beta_hedge_units(2*r, r, 10, 50, 100) == -10

## Extension
Try a different random seed or add a transaction fee. Explain how your answer changes.